```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3 done;
    class A4a current;
    class A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 04a — Lexical diversity, association, and exploratory representations (TF–IDF)

**What this notebook does**
- Loads artifacts from earlier notebooks when available (especially Session 4 OUTPUT_DIRs), with a safe fallback to re-loading `data/processed/cleaned/` + metadata.
- Computes **lexical diversity** measures (TTR variants + MTLD-lite) and **hapax** statistics.
- Computes **association** (collocations via PMI/LLR) and **keyness** (log-likelihood) across time bins.
- Builds exploratory **TF–IDF** representations (neighbors, top terms) and optionally **embeddings** (CPU-friendly) with cached OUTPUT_DIRs.

We will use libraries from scikit-learn: https://scikit-learn.org/stable/index.html.

> **Method note:** Many metrics are sensitive to document length and corpus composition. This notebook always reports document counts per bin and includes normalization / robustness checks.

## Setup

In [ ]:
!pip install scikit-learn

In [ ]:
# -----------------------------
# Import
# -----------------------------

from __future__ import annotations

from pathlib import Path
from typing import List, Tuple
import re
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Paths
# -----------------------------

PROJECT_ROOT = Path(".")  # run from the Notebooks/ folder

DATA = PROJECT_ROOT / "data"
TEXTS_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
META = PROJECT_ROOT / "analysis" / "tables" / "nb02-corpus-complete-metadata.csv"
MANUAL_YEAR = PROJECT_ROOT / "analysis" / "tables" / "nb01-manual-date-lookup.csv"

# Output and cache folders
OUTPUT_DIR = PROJECT_ROOT / "analysis"
CACHE = PROJECT_ROOT / "cache" 

# -----------------------------
# Parameters
# -----------------------------

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# -----------------------------
# Load documents' metadata
# -----------------------------
doc_index = PROJECT_ROOT / "analysis" / "tables" / "nb03-doc_index.csv"
print("\nLoading document table from:", doc_index)
df = pd.read_csv(doc_index)
df["publication_year"] = df["publication_year"].astype("Int64")
print(f"\n{len(df)} documents' metadata loaded.")

In [ ]:
df.head(2)

In [ ]:
# Save a copy for later notebooks
(df.drop(columns=[c for c in ["text"] if c in df.columns])
   .to_parquet(OUTPUT_DIR / "tables" / "nb04-docs_with_time.parquet", engine="pyarrow", index=False))

In [ ]:
# Save a copy for later notebooks
(df.to_parquet(OUTPUT_DIR / "tables" / "nb04-docs_with_time.parquet", engine="pyarrow", index=False))

# Stop words: filtering out noise before it filters out meaning

Not every word in a text carries equal weight for our analysis. Words like "the," "and," "of," or "is" appear constantly across every document, regardless of topic, period, or author — they're grammatical scaffolding, not conceptual content. When left in, these **stop words** tend to dominate frequency counts, similarity scores, and topic models, drowning out the terms we actually care about.

### What stop words are, and why a fixed list is not enough

Standard NLP libraries ship with a generic stopword list — common function words for a given language. That's a reasonable starting point, but it's built for general text, not for a 2,300-year corpus of philosophical writing. Our corpus will surface its own noise that a generic list won't catch: archaic spellings, OCR artifacts, translator's notes, recurring boilerplate from Project Gutenberg headers, or even certain high-frequency terms that turn out to be uninformative for a specific analysis (e.g. "philosophy" itself, if it appears in nearly every document).

### An evolving list

Rather than settling on one stopword list at the start and never revisiting it, we treat stopwording as an **iterative, corpus-driven process**. As we move through the notebooks — inspecting frequency tables, keyness results, topic models — we'll periodically notice terms that should be filtered out but aren't yet, and add them to a shared, growing list.

This shared list is maintained externally and loaded on demand with:

```python
STOPWORDS = load_stopwords()
```

Calling this function whenever we (re-)build a document-term matrix or vocabulary ensures every notebook is filtering against the **latest, most complete** version of the list — including additions made in earlier sessions. If you spot a term during any analysis that looks like noise rather than content, add it to the shared list, not just filter it out locally in your own notebook.

### A word of caution

Stopwording is a one-way filter: once a word is removed, it is gone from every downstream analysis that uses that vocabulary. Be conservative about what you add — a term that looks uninformative in one context (e.g. a frequency table) may turn out to be exactly the kind of term worth tracking in another (e.g. semantic drift across periods). When in doubt, discuss before adding a term that could plausibly carry conceptual meaning.

In [ ]:
STOP_WORDS_FILE = Path('./analysis/stop_words_custom.txt')

In [ ]:
def load_stopwords(filepath:Path = STOP_WORDS_FILE) -> set:
    """
    Load custom stop words from a plain text file (one per line).
    Lines starting with '#' are ignored (comments).
    """
    if not filepath.exists():
        raise FileNotFoundError(f"Stopword file not found: {filepath}")
    
    stopwords = set()
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                stopwords.add(line.lower())
    return stopwords

In [ ]:
CUSTOM_STOPWORDS = load_stopwords()
print(f"Loaded {len(CUSTOM_STOPWORDS)} custom stop words.")

# Text normalization / tokenization for lexical metrics

In [ ]:
# Parameters for tokenization / representation choices
LOWERCASE = True
MIN_TOKEN_LEN = 2 # Exclude digit and one-letter word
REMOVE_STOPWORDS = True

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

STOPWORDS = set(ENGLISH_STOP_WORDS) | CUSTOM_STOPWORDS  if REMOVE_STOPWORDS else set() # Combine scikit-learn and custom English stop words

def tokenize(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    if LOWERCASE:
        text = text.lower()
    toks = TOKEN_RE.findall(text)
    toks = [t for t in toks if len(t) >= MIN_TOKEN_LEN]
    if STOPWORDS:
        toks = [t for t in toks if t not in STOPWORDS]
    return toks

def safe_div(a, b):
    return float(a) / float(b) if b else np.nan

## Lexical diversity & hapax

We compute several diversity measures and explicitly show the document-length effect (TTR decreases as documents get longer). Prefer length-robust measures (e.g., CTTR/MTLD) for cross-document comparisons.

In [ ]:
# -----------------------------
# Helper
# -----------------------------
def read_clean_text(texts_dir: Path, pg_id: int) -> str:
    filename = f"pg{pg_id}.txt"
    return (texts_dir / filename).read_text(encoding="utf-8", errors="replace")

In [ ]:
# Create a new column and store corresponding texts
df["text"] = df["pg_id"].apply(lambda pid: read_clean_text(TEXTS_DIR, pid))

## Lexical diversity metrics (how to read them)

Lexical diversity asks: *how varied is the vocabulary in a text?* We report several complementary measures because no single metric is perfect.

- **Tokens vs types**

  **Tokens**: total running words in a document.  
  **Types**: unique word forms in the document.  
  
- **TTR (Type–Token Ratio)**
 
    $TTR = \frac{\#\text{types}}{\#\text{tokens}}$   

    Higher TTR suggests more varied vocabulary, but **TTR decreases with text length**, so it is not reliable for comparing documents of very different sizes.

- **Moving-average / windowed TTR**  

    Compute TTR on fixed-size windows (e.g., 500 tokens) and average across windows. This reduces (but does not eliminate) length sensitivity by comparing equal text windows.

- **MTLD (Measure of Textual Lexical Diversity)**  

    MTLD estimates how long a text can go before its TTR drops below a threshold. **Higher MTLD indicates more stable lexical variety** and is typically **less sensitive to length** than raw TTR.

- **Hapax legomena (and hapax rate)**  

    **Hapax** are types that occur exactly once in a document (or corpus). A high hapax rate can indicate rich or noisy vocabulary (e.g., rare terms, names, OCR artifacts), and it also increases with document length.

---  

**Note:** All lexical diversity metrics depend on preprocessing choices (lowercasing, tokenization, removing stopwords/punctuation). We therefore report document length statistics alongside diversity measures and interpret trends cautiously—especially across time bins with different numbers/lengths of texts.

In [ ]:
# -----------------------------
# Lexical diversity metrics
# -----------------------------
def ttr(tokens: list[str]) -> float:
    n = len(tokens)
    v = len(set(tokens))
    return safe_div(v, n)

def rttr(tokens: list[str]) -> float:
    # Root TTR: V / sqrt(N)
    n = len(tokens)
    v = len(set(tokens))
    return safe_div(v, math.sqrt(n)) if n else np.nan

def cttr(tokens: list[str]) -> float:
    # Corrected TTR: V / sqrt(2N)
    n = len(tokens)
    v = len(set(tokens))
    return safe_div(v, math.sqrt(2*n)) if n else np.nan

def herdan_c(tokens: list[str]) -> float:
    # Herdan's C: log(V) / log(N)
    n = len(tokens)
    v = len(set(tokens))
    if n <= 1 or v <= 1:
        return np.nan
    return math.log(v) / math.log(n)

def maas_a2(tokens: list[str]) -> float:
    # Maas measure: (log N - log V) / (log N)^2  (lower -> more diverse)
    n = len(tokens)
    v = len(set(tokens))
    if n <= 1 or v <= 1:
        return np.nan
    ln = math.log(n)
    return (ln - math.log(v)) / (ln**2)

def mtld(tokens: list[str], threshold: float = 0.72) -> float:
    # MTLD (McCarthy & Jarvis): mean length of sequential factors
    # This is a lightweight implementation; good enough for teaching.
    if len(tokens) < 50:
        return np.nan

    def factors(seq):
        types = set()
        n = 0
        f = 0
        for tok in seq:
            n += 1
            types.add(tok)
            curr_ttr = len(types) / n
            if curr_ttr <= threshold:
                f += 1
                types = set()
                n = 0
        # partial factor
        if n > 0:
            # proportion to reach threshold
            curr_ttr = len(types) / n
            if curr_ttr != 1:
                f += (1 - curr_ttr) / (1 - threshold)
        return f

    f_forward = factors(tokens)
    f_backward = factors(list(reversed(tokens)))
    f = (f_forward + f_backward) / 2
    return safe_div(len(tokens), f)

def lexical_stats(text: str) -> dict:
    toks = tokenize(text)
    n = len(toks)
    v = len(set(toks))
    hapax = sum(1 for _, c in pd.Series(toks).value_counts().items() if c == 1) if n else 0
    return {
        "n_tokens": n,
        "n_types": v,
        "ttr": ttr(toks),
        "rttr": rttr(toks),
        "cttr": cttr(toks),
        "herdan_c": herdan_c(toks),
        "maas_a2": maas_a2(toks),
        "mtld": mtld(toks),
        "hapax": hapax,
        "hapax_rate": safe_div(hapax, v),
    }

In [ ]:
# Compute diversity metrics per document
if "text" not in df.columns:
    raise ValueError("Document table must include a 'text' column.")

lex_rows = []
for row in df[["pg_id","title","publication_year","time_bin","text"]].itertuples(index=False):
    stats = lexical_stats(row.text)
    stats.update({"pg_id": row.pg_id, "title": row.title, "publication_year": row.publication_year, "time_bin": row.time_bin})
    lex_rows.append(stats)

lex = pd.DataFrame(lex_rows)
display(lex.head())

# Save tables
lex.to_parquet(OUTPUT_DIR / "tables" / "nb04-lexical_diversity_by_doc.parquet", index=False)
lex.to_csv(OUTPUT_DIR / "tables" / "nb04-lexical_diversity_by_doc.csv", index=False)

# Aggregate by time bin
by_bin = (lex.dropna(subset=["time_bin"])
            .groupby("time_bin")
            .agg(
                n_docs=("pg_id","count"),
                tokens_mean=("n_tokens","mean"),
                ttr_mean=("ttr","mean"),
                cttr_mean=("cttr","mean"),
                mtld_mean=("mtld","mean"),
                hapax_rate_mean=("hapax_rate","mean"),
            )
            .reset_index()
         )
by_bin.to_csv(OUTPUT_DIR / "tables" / "nb04-lexical_diversity_by_time_bin.csv", index=False)
display(by_bin)

In [ ]:
# Helper to extract start year from time_bin string
def get_start_year(b: str) -> int:
    try:
        return int(b.split('–')[0])
    except (ValueError, IndexError, AttributeError):
        return 0

# Compute diversity metrics per document
if "text" not in df.columns:
    raise ValueError("Document table must include a 'text' column.")

lex_rows = []
for row in df[["pg_id","title","publication_year","time_bin","text"]].itertuples(index=False):
    stats = lexical_stats(row.text)
    stats.update({"pg_id": row.pg_id, "title": row.title, "publication_year": row.publication_year, "time_bin": row.time_bin})
    lex_rows.append(stats)

lex = pd.DataFrame(lex_rows)
display(lex.head())

# Save tables
lex.to_parquet(OUTPUT_DIR / "tables" / "nb04-lexical_diversity_by_doc.parquet", index=False)
lex.to_csv(OUTPUT_DIR / "tables" / "nb04-lexical_diversity_by_doc.csv", index=False)

# Aggregate by time bin, then sort chronologically
by_bin = (lex.dropna(subset=["time_bin"])
            .groupby("time_bin")
            .agg(
                n_docs=("pg_id","count"),
                tokens_mean=("n_tokens","mean"),
                ttr_mean=("ttr","mean"),
                cttr_mean=("cttr","mean"),
                mtld_mean=("mtld","mean"),
                hapax_rate_mean=("hapax_rate","mean"),
            )
            .reset_index()
         )

# Sort by start year of the time bin
by_bin['start_year'] = by_bin['time_bin'].apply(get_start_year)
by_bin = by_bin.sort_values('start_year').drop(columns=['start_year']).reset_index(drop=True)

# Save and display
by_bin.to_csv(OUTPUT_DIR / "tables" / "nb04-lexical_diversity_by_time_bin.csv", index=False)
display(by_bin)

In [ ]:
# -----------------------------
# Plots: lexical diversity vs time and length
# -----------------------------
plt.figure(figsize=(7,4))
plt.hist(lex["n_tokens"].clip(upper=200000), bins=40, color="teal", alpha=0.85)
plt.title("Document length (tokens; clipped at 200k)")
plt.xlabel("Tokens")
plt.ylabel("Documents")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "nb04-doc_length_tokens_hist.png", dpi=150)
plt.show()

plt.figure(figsize=(7,4))
plt.scatter(lex["n_tokens"], lex["ttr"], s=10, alpha=0.35, color='teal')
plt.xscale("log")
plt.title("TTR vs document length (TTR decreases with length)")
plt.xlabel("Tokens (log scale)")
plt.ylabel("TTR")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "nb04-ttr_vs_length.png", dpi=150)
plt.show()

# Time-bin trends
plt.figure(figsize=(10,4))
plt.plot(by_bin["time_bin"].astype(str), by_bin["cttr_mean"], marker="o", color='teal')
plt.xticks(rotation=45, ha="right")
plt.title("Lexical diversity over time (Corrected TTR mean by time bin)")
plt.xlabel("Time bin")
plt.ylabel("Mean CTTR")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "nb04-cttr_over_time.png", dpi=150)
plt.show()

plt.figure(figsize=(10,4))
plt.plot(by_bin["time_bin"].astype(str), by_bin["mtld_mean"], marker="o", color="teal")
plt.xticks(rotation=45, ha="right")
plt.title("Lexical diversity over time (MTLD-lite mean by time bin)")
plt.xlabel("Time bin")
plt.ylabel("Mean MTLD")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "nb04-mtld_over_time.png", dpi=150)
plt.show()

In [ ]:
df["time_bin"].value_counts(dropna=False).head(10)

---
# Association measures: finding multiword expressions and period-specific terms

So far we have treated words individually, that is one column per term in the document-term matrix. But meaning is often carried by **combinations** of words that behave as a single unit: "human nature", "self-consciousness", "saint paul", "middle age". These are called **collocations** or **multiword expressions**, and they will not show up if we only ever look at single words in isolation.

This section introduces **association measures**, statistical tools that quantify how "unusually" two words co-occur, whether these are two adjacent words forming a bigram, or a single word and a time period.

### PMI: Pointwise Mutual Information

**PMI** asks a simple question about a pair of words: *do these two words occur together more often than we would expect by chance, given how frequent each word is individually?*

PMI(w1, w2) = log( P(w1, w2) / (P(w1) · P(w2)) )

- If two words are independent, their joint probability should just be the product of their individual probabilities, and PMI ≈ 0.
- If they co-occur *more* than chance would predict, PMI is positive and large — a strong sign the pair functions as a fixed expression rather than a coincidental pairing.

**A known limitation:** PMI is particularly sensitive to rare words. Two words that each appear only once, and happen to appear together that one time, get a very high PMI score despite there being no real evidence of a genuine collocation. This is why PMI is usually reported alongside a minimum frequency threshold, and it is rarely used alone.

### LLR: Log-Likelihood Ratio

**LLR** addresses PMI's weakness by explicitly weighing the *strength of the evidence*, not just the ratio. It compares two hypotheses statistically:

- **H₀ (independence):** the two words co-occur only as often as chance predicts.
- **H₁ (association):** the two words co-occur more (or less) than chance predicts.

LLR measures how much more likely the observed co-occurrence pattern is under H₁ than under H₀. Unlike PMI, it naturally accounts for sample size: a bigram seen 200 times has more evidential weight than one seen twice, even if both have similar raw ratios. This makes LLR generally preferred over PMI for ranking collocations in real corpora, especially when frequency varies widely across bigrams.

### From bigrams to time-bins: LLR-based keyness

The same logic extends beyond word pairs. Instead of asking "do these two *words* co-occur more than chance?", we can ask "does this *term* occur more than chance in this *time period* compared to the rest of the corpus?" This is **LLR-based keyness**: the same statistical test, applied to the pairs (term, time-bin) instead of the pairs (word, word).

This gives us a second, complementary way to detect terms characteristic of a period, one that, unlike the add-1 log-odds approach from the previous section, weighs evidence strength directly rather than relying on smoothing to handle sparse counts. Comparing the two rankings (log-odds vs. LLR-based keyness) is itself informative: terms that rank highly under both methods are the most robust candidates for genuinely period-specific vocabulary, while disagreements often point to rare or noisy terms worth a closer look.

Association measures help identify multiword expressions and terms that are unusually characteristic of a time period. We use PMI and LLR for bigram collocations, and LLR-based keyness for time bins.

In [ ]:
# Collocations / association
BIGRAM_MIN_COUNT = 10    # raise this if you get noisy collocations
TOP_K_COLLOCATIONS = 50

In [ ]:
# Build a corpus-level count model (unigrams + bigrams)
texts = df["text"].astype(str).tolist()

# Count unigram
cv_uni = CountVectorizer(
    tokenizer=tokenize, # Use the function created in helpers
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 1),
    min_df=3,
)
X_uni = cv_uni.fit_transform(texts)
uni_vocab = np.array(cv_uni.get_feature_names_out())
uni_counts = np.asarray(X_uni.sum(axis=0)).ravel()

# Count bigram
cv_bi = CountVectorizer(
    tokenizer=tokenize,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(2, 2),
    min_df=3,
)
X_bi = cv_bi.fit_transform(texts)
bi_vocab = np.array(cv_bi.get_feature_names_out())
bi_counts = np.asarray(X_bi.sum(axis=0)).ravel()

# Filter bigrams by minimum count
mask = bi_counts >= BIGRAM_MIN_COUNT
bi_vocab_f = bi_vocab[mask]
bi_counts_f = bi_counts[mask]

total_uni = uni_counts.sum()
total_bi = bi_counts_f.sum()

# Build unigram probability lookup for PMI
uni_prob = {w: c/total_uni for w, c in zip(uni_vocab, uni_counts)}

def pmi_for_bigram(bg: str, c_bg: int) -> float:
    w1, w2 = bg.split()
    p_bg = c_bg / total_bi if total_bi else 0
    p1 = uni_prob.get(w1, 1e-12)
    p2 = uni_prob.get(w2, 1e-12)
    return math.log(p_bg / (p1*p2) + 1e-12, 2)

# Dunning log-likelihood for 2x2 contingency of adjacency counts
def llr_2x2(k11, k12, k21, k22):
    def xlogx(x):
        return 0 if x == 0 else x * math.log(x)
    row1, row2 = k11 + k12, k21 + k22
    col1, col2 = k11 + k21, k12 + k22
    total = row1 + row2
    return 2 * (
        xlogx(k11) + xlogx(k12) + xlogx(k21) + xlogx(k22)
        - xlogx(row1) - xlogx(row2) - xlogx(col1) - xlogx(col2)
        + xlogx(total)
    )

# Approximate contingency for bigram w1 w2:
# k11 = count(w1 w2)
# k12 = count(w1 *) - k11
# k21 = count(* w2) - k11
# k22 = total_bi - k11 - k12 - k21
# We estimate count(w1 *) and count(* w2) from bigram marginal sums computed from filtered bigrams.
w1_marg = {}
w2_marg = {}
for bg, c in zip(bi_vocab_f, bi_counts_f):
    w1, w2 = bg.split()
    w1_marg[w1] = w1_marg.get(w1, 0) + c
    w2_marg[w2] = w2_marg.get(w2, 0) + c

rows = []
for bg, c in zip(bi_vocab_f, bi_counts_f):
    w1, w2 = bg.split()
    k11 = int(c)
    k12 = int(w1_marg.get(w1, 0) - k11)
    k21 = int(w2_marg.get(w2, 0) - k11)
    k22 = int(total_bi - k11 - k12 - k21)
    rows.append({
        "bigram": bg,
        "count": k11,
        "pmi": pmi_for_bigram(bg, k11),
        "llr": llr_2x2(k11, k12, k21, k22)
    })

colloc = pd.DataFrame(rows).sort_values(["llr","pmi","count"], ascending=False)
colloc_top = colloc.head(TOP_K_COLLOCATIONS)
display(colloc_top)

### Raw collocation rankings (PMI/LLR) often surface “junk” bigrams:
- stopword-driven pairs: "of the", "in a", ...
- very short tokens / abbreviations: "ab cd", ...
- corpus-specific boilerplate tokens

The following cell filters bigrams _after_ extraction so you can iteratively improve
the results: inspect the top collocations table, then add/remove items from the stop_words_custom.txt 
file in `./analysis/stop_words_custom.txt` and update the stop words list.

# Fill the gap
### _Collocations cleanup: filter stop_words + short tokens_

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Update the list of custom stop words in `./analysis/stop_words_custom.txt`
# Write the code below to update STOPWORDS using scikit-learn ENGLISH_STOP_WORDS and the custom list
CUSTOM_STOPWORDS =                # Update your custom list if necessary using the function defined at the beginning of the notebook
print(f"Loaded {len(CUSTOM_STOPWORDS)} custom stop words.")

STOPWORDS =                       # Assign to the variable `STOPWORDS` the union of ENGLISH_STOP_WORDS and custom stop words.
print(f"Loaded {len(STOPWORDS)} stop words.")

In [ ]:
MIN_TOKEN_LEN = 3  # can be raised to 4 if too many noisy short tokens remain

def keep_bigram(bg: str) -> bool:
    parts = bg.split()
    if len(parts) != 2:
        return False
    w1, w2 = parts
    if (w1 in STOPWORDS) or (w2 in STOPWORDS):
        return False
    if (len(w1) < MIN_TOKEN_LEN) or (len(w2) < MIN_TOKEN_LEN):
        return False
    return True

# Apply to your filtered bigram vocab/counts arrays (bi_vocab_f, bi_counts_f)
mask = np.array([keep_bigram(bg) for bg in bi_vocab_f], dtype=bool)
bi_vocab_f2 = bi_vocab_f[mask]
bi_counts_f2 = bi_counts_f[mask]

print(f"Filtered bigrams: kept {len(bi_vocab_f2):,} / {len(bi_vocab_f):,} ({len(bi_vocab_f2)/max(len(bi_vocab_f),1):.1%})")

In [ ]:
# -----------------------------
# Association: collocations (bigrams) via PMI and log-likelihood ratio (LLR)
# -----------------------------

# Check what the change in the stopwords produces
bi_vocab_f = bi_vocab_f2 # New bigram vocab after extra stopwords removal
bi_counts_f = bi_counts_f2 # New bigram counts after extra stopwords removal

total_uni = uni_counts.sum()
total_bi = bi_counts_f.sum()

# Build unigram probability lookup for PMI
uni_prob = {w: c/total_uni for w, c in zip(uni_vocab, uni_counts)}

# Approximate contingency for bigram w1 w2:
# k11 = count(w1 w2)
# k12 = count(w1 *) - k11
# k21 = count(* w2) - k11
# k22 = total_bi - k11 - k12 - k21
# We estimate count(w1 *) and count(* w2) from bigram marginal sums computed from filtered bigrams.
w1_marg = {}
w2_marg = {}
for bg, c in zip(bi_vocab_f, bi_counts_f):
    w1, w2 = bg.split()
    w1_marg[w1] = w1_marg.get(w1, 0) + c
    w2_marg[w2] = w2_marg.get(w2, 0) + c

rows = []
for bg, c in zip(bi_vocab_f, bi_counts_f):
    w1, w2 = bg.split()
    k11 = int(c)
    k12 = int(w1_marg.get(w1, 0) - k11)
    k21 = int(w2_marg.get(w2, 0) - k11)
    k22 = int(total_bi - k11 - k12 - k21)
    rows.append({
        "bigram": bg,
        "count": k11,
        "pmi": pmi_for_bigram(bg, k11),
        "llr": llr_2x2(k11, k12, k21, k22)
    })

colloc = pd.DataFrame(rows).sort_values(["llr","pmi","count"], ascending=False)
colloc_top = colloc.head(TOP_K_COLLOCATIONS)
display(colloc_top)

In [ ]:
# Save to CSV
colloc.to_csv(OUTPUT_DIR / "tables" / "nb04-collocations_bigrams_all.csv", index=False)
colloc_top.to_csv(OUTPUT_DIR / "tables" / "nb04-collocations_bigrams_top.csv", index=False)

In [ ]:
# -----------------------------
# Keyness: log-likelihood (Dunning) per time bin vs rest of corpus
# -----------------------------
def log_likelihood_keyness(k_target, n_target, k_ref, n_ref):
    # 2x2 counts:
    # k11 = k_target, k12 = n_target - k_target
    # k21 = k_ref,    k22 = n_ref - k_ref
    return llr_2x2(k_target, n_target - k_target, k_ref, n_ref - k_ref)

# Build a document-term matrix for unigrams (sparse)
cv_kw = CountVectorizer(tokenizer=tokenize, token_pattern=None, ngram_range=(1,1), min_df=3, max_df=0.8)
X = cv_kw.fit_transform(df["text"].astype(str).tolist())
vocab = np.array(cv_kw.get_feature_names_out())
term_counts_all = np.asarray(X.sum(axis=0)).ravel()
n_all = term_counts_all.sum()

# Compute per-bin keyness
key_rows = []
for bin_label, idx in df.dropna(subset=["time_bin"]).groupby("time_bin").indices.items():
    X_t = X[idx]
    counts_t = np.asarray(X_t.sum(axis=0)).ravel()
    n_t = counts_t.sum()
    counts_r = term_counts_all - counts_t
    n_r = n_all - n_t

    llr_scores = np.array([log_likelihood_keyness(int(ct), int(n_t), int(cr), int(n_r))
                           for ct, cr in zip(counts_t, counts_r)])
    # direction: overuse in target vs rest
    # log ratio as sign (avoid zeros)
    p_t = (counts_t + 0.5) / (n_t + 1.0)
    p_r = (counts_r + 0.5) / (n_r + 1.0)
    log_ratio = np.log(p_t / p_r)
    signed_llr = llr_scores * np.sign(log_ratio)

    top_idx = np.argsort(-signed_llr)[:30]
    for j in top_idx:
        key_rows.append({
            "time_bin": str(bin_label),
            "term": vocab[j],
            "llr_signed": float(signed_llr[j]),
            "count_bin": int(counts_t[j]),
            "count_rest": int(counts_r[j]),
        })

key = pd.DataFrame(key_rows)
display(key.head(20))

# LLR-based Keyness 

Workflow:
1) Run once the heavy parts (vectorize + LLR)
2) Update `STOPWORDS` based on the printed candidates/table
3) Rerun to update filtering


In [ ]:
# ----------------------------
# (1) RUN ONCE
# ----------------------------

# ----------------------------
# Parameters (vectorization)
# ----------------------------
MIN_DF = 3
MAX_DF = 0.8

# ----------------------------
# Filtering knobs (rerunnable)
# ----------------------------
MIN_TOKEN_LEN = 3
MIN_COUNT_BIN = 5
MIN_COUNT_TOTAL = 20
TOP_K_PER_BIN = 30

# ----------------------------
# Helper (define once)
# ----------------------------
def keep_term(t: str, stop: set[str], min_token_len: int = 3) -> bool:
    t = str(t).strip().lower()
    if not t or t in stop:
        return False
    if len(t) < min_token_len:
        return False
    if re.fullmatch(r"\d+", t):
        return False
    if not re.search(r"[a-z]", t):
        return False
    return True

def log_likelihood_keyness(k_target: int, n_target: int, k_ref: int, n_ref: int) -> float:
    return llr_2x2(k_target, n_target - k_target, k_ref, n_ref - k_ref)

if "text" not in df.columns:
    raise KeyError("df must contain a 'text' column for keyness computation.")
if "time_bin" not in df.columns:
    raise KeyError("df must contain a 'time_bin' column for per-bin keyness.")

df_bins = df.dropna(subset=["time_bin"]).copy()

cv_kw = CountVectorizer(
    tokenizer=tokenize,      
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 1),
    min_df=MIN_DF,
    max_df=MAX_DF,
)

texts = df_bins["text"].astype(str).tolist()
X = cv_kw.fit_transform(texts)

vocab = np.array(cv_kw.get_feature_names_out())
term_counts_all = np.asarray(X.sum(axis=0)).ravel().astype(np.int64)
n_all = int(term_counts_all.sum())

# Compute LLR per time bin vs rest
key_rows = []
groups = df_bins.groupby("time_bin").indices

for bin_label, idx in groups.items():
    idx = list(idx)
    X_t = X[idx]
    counts_t = np.asarray(X_t.sum(axis=0)).ravel().astype(np.int64)
    n_t = int(counts_t.sum())

    counts_r = term_counts_all - counts_t
    n_r = int(n_all - n_t)

    llr_scores = np.array(
        [log_likelihood_keyness(int(ct), int(n_t), int(cr), int(n_r)) for ct, cr in zip(counts_t, counts_r)],
        dtype=float
    )

    # signed direction using smoothed log ratio
    p_t = (counts_t + 0.5) / (n_t + 1.0)
    p_r = (counts_r + 0.5) / (n_r + 1.0)
    signed_llr = llr_scores * np.sign(np.log(p_t / p_r))

    # keep extra, filtering will remove some
    top_idx = np.argsort(-signed_llr)[: max(5 * TOP_K_PER_BIN, TOP_K_PER_BIN)]

    for j in top_idx:
        key_rows.append({
            "time_bin": str(bin_label),
            "term": str(vocab[j]),
            "llr_signed": float(signed_llr[j]),
            "count_bin": int(counts_t[j]),
            "count_rest": int(counts_r[j]),
            "count_total": int(counts_t[j] + counts_r[j]),
        })

# Store raw keyness results in a global variable for fast re-filtering
# "Raw keyness rows" = how many (time_bin, term) keyness entries we computed BEFORE filtering.
# Each row is one candidate “key” term for one bin; we keep more than TOP_K initially because
# stopword/low-count filtering will remove many rows later.
key = pd.DataFrame(key_rows)

print("Raw keyness rows:", f"{len(key):,}")

In [ ]:
# Diagnostic: suggest short/noisy candidates from the raw table
cand = (
    key.assign(term_l=key["term"].str.lower())
       .loc[lambda d: d["term_l"].str.len().between(1, 4)]
       .groupby("term_l")["count_total"].sum()
       .sort_values(ascending=False)
       .head(30)
)
print("\nCandidate short/noisy terms to consider adding to CUSTOM_STOPWORDS:")
print(cand.to_string())

# Fill the gap

- Update the list of custom stop words in `./analysis/stop_words_custom.txt`
- Write the code below to update STOPWORDS using scikit-learn ENGLISH_STOP_WORDS and the custom list

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Update the list of custom stop words in `./analysis/stop_words_custom.txt`
# Write the code below to update STOPWORDS using scikit-learn ENGLISH_STOP_WORDS and the custom list
CUSTOM_STOPWORDS =                # Update your custom list if necessary using the function defined at the beginning of the notebook
STOPWORDS =                       # Assign to the variable `STOPWORDS` the union of ENGLISH_STOP_WORDS and custom stop words.
print(f"Loaded {len(STOPWORDS)} stop words.")

In [ ]:
# ----------------------------
# EDIT + RERUN
# ----------------------------
if "key" not in globals():
    raise NameError("`key` not found. First set RUN_ONCE=True and run this cell once.")

# Apply filtering
mask = (
    key["term"].apply(lambda x: keep_term(x, STOPWORDS, MIN_TOKEN_LEN))
    & (key["count_bin"] >= MIN_COUNT_BIN)
    & (key["count_total"] >= MIN_COUNT_TOTAL)
)

key_filt = key.loc[mask].copy()

# Top-K per bin after filtering
key_top = (
    key_filt.sort_values(["time_bin", "llr_signed"], ascending=[True, False])
            .groupby("time_bin", as_index=False)
            .head(TOP_K_PER_BIN)
            .reset_index(drop=True)
)

print(f"\nFiltered keyness: top {TOP_K_PER_BIN} per bin after filtering | rows={len(key_top):,}")
display(key_top.head(30))

In [ ]:
# Save output to CSV
key.to_csv(OUTPUT_DIR / "tables" / "nb04-keyness_top_terms_by_time_bin.csv", index=False)

# Exploratory representations

## _TF–IDF (Term Frequency–Inverse Document Frequency)_

So far, we have mostly worked with **raw counts** (how often a word appears). Raw counts are useful, but they are dominated by very common words and by document length. **TF–IDF** is a simple weighting scheme that shifts attention toward words that are **characteristic of a document (or time bin)** rather than merely frequent in the whole corpus.

Intuition:
- **TF (term frequency)**: words that occur many times in a document matter more *for that document*.
- **IDF (inverse document frequency)**: words that appear in *many* documents are less informative; words that appear in *few* documents are more distinctive.

In practice, TF–IDF often highlights content-bearing terms (e.g., *virtue, reason, sovereignty*) while down-weighting ubiquitous words (e.g., *the, and, of*). In this notebook we use TF–IDF as an **exploratory representation**:
- to inspect “top terms” that characterize documents or periods,
- to compute similarity between documents (nearest neighbors),
- and to compare its behavior with embedding-based similarity later on.

**Note:** TF–IDF is still a bag-of-words model: it does not understand syntax and it does not capture meaning directly. Results are sensitive to preprocessing (tokenization, stopwords, n-grams) and to corpus composition (which texts are included in the comparison set).

## _Finding Neighboring Documents with Cosine Similarity_

Once each document has been turned into a vector — whether from TF-IDF or from embeddings — we can ask a very natural question: **which other documents in the corpus are "closest" to this one in meaning?**

### Why cosine, not raw distance

A document vector's *direction* reflects what it's about; its *length* often just reflects incidental factors like document size or overall word frequency (a longer text tends to produce larger raw counts, regardless of topic). **Cosine similarity** measures the angle between two document vectors while ignoring their length, so two documents get a high score because they emphasize similar terms in similar proportions — not simply because they are both long.

cos(θ) = (a · b) / (‖a‖ · ‖b‖)

The result is a single score between -1 and 1 (in practice, usually between 0 and 1 for count- or TF-IDF-based vectors, since term counts cannot be negative):

- **Close to 1** → the two documents point in nearly the same direction — highly similar content.
- **Close to 0** → the two documents share almost nothing distinctive.

### From pairwise similarity to "neighbors"

To find a document's **nearest neighbors**, we compute its cosine similarity against every other document in the corpus, then sort and keep the highest-scoring matches. Doing this for all documents at once produces a full similarity matrix — a table where cell (i, j) tells us how similar document i is to document j.

```python
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(X)  # X = document vectors (TF-IDF or embeddings)

# Top 5 neighbors for a given document, excluding itself
doc_idx = 0
scores = sim_matrix[doc_idx]
top_neighbors = scores.argsort()[::-1][1:6]  # skip index 0 = itself
```

### Why this matters for our corpus

Nearest-neighbor search is a first, intuitive way to explore the semantic network we have been building in this course: it lets us ask concrete questions like *"which texts most resemble this one?"* or *"does this text's nearest neighbor come from the same period, school, or author — or does it cross those boundaries?"* Neighbors that cross expected boundaries (e.g. a 17th-century text closest to a 3rd-century one) are often the most interesting cases, since they hint at conceptual continuity that a purely chronological or biographical reading of the corpus would miss.

In [ ]:
# -----------------------------
# Exploratory TF–IDF: neighbors + top terms per bin
# -----------------------------
# Goal of this section:
# 1) Build a TF–IDF representation of each document (each book = one vector).
# 2) Use TF–IDF vectors to:
#    - inspect which terms are most characteristic of a single document
#    - find nearest-neighbor documents by cosine similarity
#    - compute "signature terms" for each time bin by averaging TF–IDF vectors in that bin

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse

In [ ]:
# -----------------------------
# Helper: top TF–IDF terms for one document
# -----------------------------
def top_terms_for_doc(i: int, k: int = 15) -> List[Tuple[str, float]]:
    """
    Extract the top-k terms with highest TF‑IDF weight for a specific document.

    The function retrieves the TF‑IDF vector for document `i` from the global
    TF‑IDF matrix `T`, sorts the non‑zero terms by their weight in descending
    order, and returns the top `k` terms along with their TF‑IDF values.

    Parameters
    ----------
    i : int
        The row index of the document in the DataFrame `df` and in the
        TF‑IDF matrix `T` (must be 0 ≤ i < T.shape[0]).
    k : int, default 15
        The number of top terms to return. If the document has fewer than
        `k` non‑zero terms, only those terms are returned (no padding).

    Returns
    -------
    List[Tuple[str, float]]
        A list of `(term, tfidf_weight)` tuples, sorted from highest to
        lowest weight. For example:
        [('philosophy', 0.892), ('ethics', 0.781), ('moral', 0.654)]

        If the document has no non‑zero TF‑IDF terms (i.e., all weights are
        zero), an empty list is returned.

    Notes
    -----
    - The function uses the raw TF‑IDF weights as stored in `T` (no additional
      normalisation is applied).
    """
    row = T.getrow(i)               # sparse row vector for doc i
    if row.nnz == 0:                # nnz = number of non-zero entries
        return []
    order = np.argsort(-row.data)[:k]   # row.data = corresponding tf-idf weights
    idx = row.indices[order]            # row.indices = term indices in doc
    return list(zip(terms[idx], row.data[order]))

In [ ]:
# -----------------------------
# Helper: nearest neighbors for one document
# -----------------------------
def doc_neighbors(i: int, k: int = 5) -> List[Tuple[int, float, int, str]]:
    """
    Find the k most similar documents to a given document using cosine similarity.

    The function computes the cosine similarity between the TF‑IDF vector of
    document `i` and all other documents in the corpus (including itself).
    It then returns the top `k` most similar documents (excluding the query
    document itself, which would have similarity = 1.0).

    Parameters
    ----------
    i : int
        The row index of the query document in the TF‑IDF matrix `T` and in
        the DataFrame `df` (must be 0 ≤ i < T.shape[0]).
    k : int, default 5
        The number of nearest neighbours to return (excluding the query
        document itself). If `k` is larger than the number of other documents,
        only the available documents are returned.

    Returns
    -------
    List[Tuple[int, float, int, str]]
        A list of tuples, each containing:
        - `int`: the row index of the neighbour document in `T`/`df`.
        - `float`: the cosine similarity score between the query and the neighbour.
        - `int`: the `pg_id` (Project Gutenberg ID) of the neighbour document.
        - `str`: the title of the neighbour document.

        The list is sorted from highest to lowest similarity. The query document
        itself is **excluded** from the returned list (unless you modify the
        function to include it).

    Notes
    -----
    - This function relies on the global variables `T` (the TF‑IDF sparse matrix)
      and `df` (the DataFrame containing document metadata) being defined in the
      notebook environment.
    - `T` must be a `scipy.sparse.csr_matrix` where each row corresponds to a
      document and each column to a term.
    - The function computes similarities using `cosine_similarity` from
      scikit‑learn. This may be memory‑intensive for large corpora; consider
      using `T[i]` as a sparse vector.
    - The query document is included in the initial `k+1` neighbours, but it is
      skipped in the output (the first neighbour in the sorted list is the
      query itself).

    Examples
    --------
    >>> Assuming T and df are defined:
    >>> neighbors = doc_neighbors(i=0, k=3)
    >>> for idx, sim, pg_id, title in neighbors:
    ...     print(f"Doc {pg_id} ({title}): similarity = {sim:.4f}")
    Doc 10108 (pg10108.txt): similarity = 0.8921
    Doc 10112 (pg10112.txt): similarity = 0.7814
    Doc 1016 (pg1016.txt): similarity = 0.6542

    >>> If you want to see the query document itself (for debugging):
    >>> Modify the function to return k+1 neighbours including the query.

    >>> If k is larger than the number of documents:
    >>> doc_neighbors(i=0, k=1000)  # returns all other documents
    """
    # Compute cosine similarity between document i and all documents
    sims = cosine_similarity(T[i], T).ravel()  # array of shape (n_docs,)

    # Sort descending; first index is the query document itself (similarity = 1)
    nn = np.argsort(-sims)[:k+1]  # take k+1 to include the query, then drop it

    out = []
    for j in nn:
        # Skip the query document itself (the first one, since similarity = 1.0)
        if j == i:
            continue
        out.append((
            int(j),                      # row index in T/df
            float(sims[j]),              # cosine similarity
            int(df.iloc[j]["pg_id"]),    # Gutenberg ID
            df.iloc[j]["title"]          # title
        ))
        if len(out) >= k:  # stop after collecting k neighbours
            break

    return out

### Build TF–IDF representation

In [ ]:
# -----------------------------
# Parameters
# -----------------------------

TFIDF_MAX_FEATURES = 50000
TFIDF_MIN_DF = 3
TFIDF_NGRAM_RANGE = (1, 2)


## TF-IDF configuration: what our parameters mean

Before building the TF-IDF matrix, we fix a few key parameters that shape what vocabulary gets included and how. Each one is a trade-off between richness of representation and computational/statistical practicality.

```python
TFIDF_MAX_FEATURES = 50000
TFIDF_MIN_DF = 3
TFIDF_NGRAM_RANGE = (1, 2)
```

### `TFIDF_MAX_FEATURES = 50000`

This caps the vocabulary at the **50,000 highest-frequency terms** across the corpus; anything beyond that rank is dropped. Without a cap, a corpus of 572 texts spanning 2,300 years could easily produce a vocabulary in the hundreds of thousands once rare spellings, proper nouns, and OCR noise are counted — most of it uninformative. Capping the vocabulary keeps the matrix computationally manageable and avoids sparse, unreliable columns of near-unique terms.

### `TFIDF_MIN_DF = 3`

`min_df` sets a **minimum document frequency**: a term must appear in at least 3 *different documents* to be included at all (not 3 times total — 3 separate documents). This filters out terms that occur in only one or two texts, which are usually one of two things: idiosyncratic to a single author or translation, or artifacts of OCR errors and archaic spelling variants. Requiring a term to recur across multiple documents is a simple, effective way to keep the vocabulary focused on terms that could plausibly represent a shared concept rather than a one-off.

### `TFIDF_NGRAM_RANGE = (1, 2)`

This tells the vectorizer to include both **unigrams** (single words, e.g. "reason") and **bigrams** (two-word sequences, e.g. "state of nature" → "state_of", "of_nature" as constituent bigrams, or more precisely adjacent pairs like "natural_law"). Many philosophical concepts are genuinely multiword — "human nature", "saint paul", "social contract" — and treating them only as separate unigrams loses exactly the compositional meaning we are trying to capture. Including bigrams lets the vocabulary represent these fixed expressions directly, at the cost of a larger, sparser vocabulary (which is part of why `max_features` and `min_df` matter — they keep this expansion in check).

### Why these three work together

These parameters are not independent choices — they interact. Allowing bigrams (`ngram_range`) inflates the raw vocabulary size considerably, which is exactly what `min_df` and `max_features` are there to control. If you change one (e.g. extend to trigrams, or lower `min_df` to 2), expect the other two to need revisiting to keep the resulting matrix at a similar scale and quality.

In [ ]:
# NOTE: We use tokenizer=tokenize (our custom tokenizer), so we set token_pattern=None
# and lowercase=False because tokenize() already handles casing consistently.
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize,                # custom tokenizer defined above
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=TFIDF_NGRAM_RANGE,     # e.g., (1,2) for unigrams + bigrams
    min_df=TFIDF_MIN_DF,               # ignore extremely rare terms
    max_features=TFIDF_MAX_FEATURES,   # cap vocabulary for memory/speed stability
    sublinear_tf=True,                 # use 1 + log(tf) to reduce dominance of repeated terms
    norm="l2",                         # makes cosine similarity meaningful
)

# T is a sparse matrix of shape (n_docs, n_terms)
T = tfidf_vec.fit_transform(df["text"].astype(str).tolist())
terms = np.array(tfidf_vec.get_feature_names_out())

print("TF–IDF matrix shape:", T.shape)
print("Vocabulary size:", len(terms))

# -----------------------------
# Save TF–IDF artifacts for later notebooks (no recomputation)
# -----------------------------
# We save:
# - the sparse TF–IDF matrix for all documents
# - the vocabulary (term list) so column indices can be interpreted later
sparse.save_npz(CACHE / "nb04-tfidf_docs.npz", T)
pd.Series(terms).to_csv(CACHE / "nb04-tfidf_terms.csv", index=False)
print(f"\nOutput saved to {CACHE / 'nb04-tfidf_terms.csv'}")

In [ ]:
# -----------------------------
# Demonstration: pick a random doc and inspect its TF–IDF profile + neighbors
# -----------------------------
i0 = np.random.randint(0, len(df))
print("Example doc metadata:", df.iloc[i0][["pg_id","title","publication_year","time_bin"]].to_dict())

print("\nTop TF–IDF terms (this document):")
print(top_terms_for_doc(i0, k=15))

print("\nNearest neighbors (TF–IDF cosine similarity):")
display(pd.DataFrame(doc_neighbors(i0, k=8), columns=["row","cosine","pg_id","title"]))

# -----------------------------
# Top terms per time bin (mean TF–IDF)
# -----------------------------
# Idea:
# - Each document is a TF–IDF vector.
# - For a given time bin, average the vectors of documents in that bin (centroid).
# - The largest coordinates in the centroid correspond to terms most characteristic of that period.
bin_terms = []

for bin_label, idx in df.dropna(subset=["time_bin"]).groupby("time_bin").indices.items():
    # Average TF–IDF vector across docs in the bin
    mean_vec = np.asarray(T[idx].mean(axis=0)).ravel()

    # Take top terms by mean weight
    top = np.argsort(-mean_vec)[:30]

    for j in top:
        bin_terms.append({
            "time_bin": str(bin_label),
            "term": terms[j],
            "mean_tfidf": float(mean_vec[j]),
        })

bin_terms = pd.DataFrame(bin_terms)

print("\nExample: top terms per bin (first rows)")
display(bin_terms.head(20))

### _How can we visualise our output?_

In [ ]:
# Helper to extract start year from time_bin string
# and order time bins chronologically
def get_start_year(b: str) -> int:
    try:
        # Handle both en dash '–' and hyphen '-'
        return int(b.split('–')[0] if '–' in b else b.split('-')[0])
    except (ValueError, IndexError, AttributeError):
        return 0

In [ ]:
import seaborn as sns
# Pivot: rows=terms, cols=time bins, values=mean TF–IDF
H = bin_terms.pivot_table(index="term", columns="time_bin", values="mean_tfidf", fill_value=0.0)

# Sort columns chronologically by start year
chronological_bins = sorted(H.columns, key=get_start_year)
H = H[chronological_bins]   # reorder columns

# Keep only the most informative terms overall
top_terms = H.max(axis=1).sort_values(ascending=False).head(50).index
H = H.loc[top_terms]

plt.figure(figsize=(12, 10))
sns.heatmap(H, cmap="crest")
plt.title("Mean TF–IDF by time bin (top terms per bin)")
plt.xlabel("Time bin")
plt.ylabel("Term")
plt.tight_layout()
plt.show()

In [ ]:
# ---- CONFIGURATION ----
# Option 1: Load custom stopwords from a file (one per line)
# STOPWORD_FILE = Path("resources/stopwords_custom.txt")
# if STOPWORD_FILE.exists():
#     with open(STOPWORD_FILE, 'r', encoding='utf-8') as f:
#         EXCLUDED_TERMS = {line.strip().lower() for line in f if line.strip() and not line.startswith('#')}
# else:
#     EXCLUDED_TERMS = set()

# Option 2: Directly define a list of terms to exclude
EXCLUDED_TERMS = {"kai",
                  "nai",
                  "pist",
                  "phu",
                  "kei",
                  "tai",
                  "stin",
                  "nthr",
                  "teron",
                  "ahura",
                  "lysias",
                  "chi"
    # Add more custom terms here
}

# ---- Filter bin_terms before pivoting ----
# Remove rows where term is in EXCLUDED_TERMS (case-insensitive)
bin_terms_filtered = bin_terms[~bin_terms["term"].str.lower().isin(EXCLUDED_TERMS)]

# Pivot: rows=terms, cols=time bins, values=mean TF–IDF
H = bin_terms_filtered.pivot_table(index="term", columns="time_bin", values="mean_tfidf", fill_value=0.0)

# Sort columns chronologically by start year
chronological_bins = sorted(H.columns, key=get_start_year)
H = H[chronological_bins]   # reorder columns

# Keep only the most informative terms overall
top_terms = H.max(axis=1).sort_values(ascending=False).head(50).index
H = H.loc[top_terms]

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(H, cmap="crest")
plt.title("Mean TF–IDF by time bin (top terms per bin, stopwords removed)")
plt.xlabel("Time bin")
plt.ylabel("Term")
plt.tight_layout()
plt.show()

In [ ]:
# Save output to CSV
bin_terms_filtered.to_csv(OUTPUT_DIR / "tables" / "nb04-tfidf_top_terms_by_time_bin.csv", index=False)
# Uncomment below if you want all bin_terms to be considered
# bin_terms.to_csv(OUTPUT_DIR / "tables" / "nb04-tfidf_top_terms_by_time_bin.csv", index=False)

In [ ]:
def bin_centroid(bin_label):
    idx = df.index[df["time_bin"].astype(str) == str(bin_label)].to_numpy()
    v = np.asarray(T[idx].mean(axis=0)).ravel()
    v = v / (np.linalg.norm(v) + 1e-12)
    return v

def compare_time_bin(bin_a, bin_b, top_k=20):
    u = bin_centroid(bin_a)
    v = bin_centroid(bin_b)
    contrib = u * v                      # per-term contribution to cosine similarity
    top = np.argsort(-contrib)[:top_k]
    return [(terms[i], float(contrib[i])) for i in top]

In [ ]:
for i, tb in enumerate(df.time_bin.unique()):
    print(i, ':', tb)

In [ ]:
compare_time_bin(df.time_bin.unique()[3], df.time_bin.unique()[2])

---

# TF–IDF document map (2D projection with TruncatedSVD)

Goal:
Visualize documents in a reduced 2D TF–IDF space (documents that appear close together have more similar TF–IDF profiles).

## Visualizing Documents with TruncatedSVD

Our TF-IDF matrix has one row per document and up to 50,000 columns — one per term. It is sparse and high-dimensional. It is contains far too many dimensions to be plotted or visually inspected: humans can only really look at 2 (or maybe 3) dimensions at once. To actually *see* the corpus, we need to compress it down to just 2 dimensions while preserving as much of the meaningful structure as possible. This is what **dimensionality reduction** does. To reduce dimensionality, we will use a technique called **TruncatedSVD**.

### What TruncatedSVD does

**SVD** stands for **Singular Value Decomposition**, a method from linear algebra for breaking a matrix down into simpler pieces. Applied to our document-term matrix, it identifies a small number of underlying **components** — patterns of terms that tend to rise and fall together across documents (for instance, one component might capture a mix of ethics-related vocabulary, another a mix of logic-related vocabulary). Instead of describing each document using all 50,000 original term-columns, we can approximately describe it using just a few of these components.

We keep only the two components that capture the **most variance** — the two patterns that best distinguish documents from one another — and discard the rest. "**Truncated**" refers to this deliberate cutoff: rather than computing the full decomposition (which would have as many components as the original matrix), we stop early and keep only the handful of components we actually want to plot.

### A quick note on PCA, and why we do not use it here

You may have already encountered, or will later encounter, **PCA (Principal Component Analysis)** — the most commonly taught dimensionality-reduction method, widely used across data science. PCA and SVD are closely related mathematically and often produce similar results in spirit: both look for the directions that best capture how the data varies.

The difference that matters for us: PCA requires **centering** the data first (subtracting the average value from every column), which turns a sparse matrix — one that is almost entirely filled with zeros, like our TF-IDF matrix — into a dense one, where nearly every entry becomes non-zero. For a matrix with 50,000 columns, converting to dense form can be prohibitively slow or memory-intensive. **TruncatedSVD works directly on the sparse matrix**, skipping the centering step entirely, which makes it the standard, practical choice specifically for sparse text data like ours.

### LSA: a name you may see elsewhere

When TruncatedSVD is applied specifically to a TF-IDF (or similar term-frequency) matrix, this exact technique has a well-established name in the NLP literature: **Latent Semantic Analysis (LSA)** (sometimes "Latent Semantic Indexing"). If you read papers or documentation that mention LSA, know that it refers to what we are doing in this section — TruncatedSVD applied to text data. "Latent" here means *hidden*: the components represent underlying patterns of meaning that are not directly observable in the raw text, only inferred statistically from patterns of word co-occurrence.

### What the resulting 2D plot means

Each document becomes a single point, positioned according to its value on the two components we kept. Distance between points in this space approximates similarity in vocabulary usage:

- **Clusters** — tight groups of points are documents that draw on a similar vocabulary, which may or may not align with what you would expect from period, author, or traditions.
- **Overlap** — where two expected groups (e.g. two periods, or two subcorpora) blend together in the plot, that is a visual signal of shared vocabulary between them.
- **Isolated points** — documents sitting apart from everything else are using distinctive vocabulary relative to the rest of the corpus. Worth a closer look — these could be genuinely unusual texts, or artifacts (translation quirks, OCR issues, unusually short documents).

### A caution about interpretation

Two dimensions is a drastic compression from 50,000 — enough to reveal broad structure, but it necessarily discards a great deal of nuance. Two documents that appear close in this plot are similar with respect to the *dominant* patterns of vocabulary variation in the corpus, but may still differ substantially in ways the top two components do not capture. Treat this plot as a starting point for exploration, not a definitive map of document similarity.

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Sample documents for readability 
MAX_DOCS_PLOT = len(df)

plot_idx = np.arange(len(df))
if len(plot_idx) > MAX_DOCS_PLOT:
    rng = np.random.default_rng(42)
    plot_idx = rng.choice(plot_idx, size=MAX_DOCS_PLOT, replace=False)

# Reduce TF–IDF vectors to 2 dimensions
svd = TruncatedSVD(n_components=2, random_state=42)
Z = svd.fit_transform(T[plot_idx])

plot_df = df.iloc[plot_idx].copy()
plot_df["svd1"] = Z[:, 0]
plot_df["svd2"] = Z[:, 1]

# Get unique time bins from plot_df and sort chronologically
time_bins = sorted(plot_df["time_bin"].dropna().unique(), key=get_start_year)

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x="svd1",
    y="svd2",
    hue="time_bin",
    hue_order=time_bins,          # set chronological order
    alpha=0.8,
    s=60,
)

plt.title("Documents in TF–IDF space (TruncatedSVD 2D)")
plt.xlabel("SVD component 1")
plt.ylabel("SVD component 2")
plt.legend(title="Time bin", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(
    "Explained variance ratio (2 components):",
    round(float(svd.explained_variance_ratio_.sum()), 3)
)

### Companion table for the TruncatedSVD projection

In [ ]:
# ======================================================================
# Goal:
# Link the 2D TF–IDF map back to actual corpus documents to facilitate
# the interpretation of clusters/outliers through titles and metadata.
# ======================================================================

# Build a readable projection table
svd_table = plot_df[["pg_id", "title", "publication_year", "svd1", "svd2"]]
svd_table["publication_year"] = pd.to_numeric(svd_table["publication_year"], errors="coerce").astype("Int64")

# Sort for easier inspection
if "time_bin" in svd_table.columns:
    svd_table = svd_table.sort_values(["time_bin", "svd1"])

print("Projection table (documents in TF–IDF space):")
display(svd_table.head(30))

# Optional: save for later inspection
# svd_table.to_csv(ANALYSIS_TABLES_DIR / "nb04_tfidf_svd_projection_table.csv", index=False)

### Interactive TF–IDF document map with Plotly

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"   # or "browser", "vscode", etc.

In [ ]:
# ============================================================
# Interactive TF–IDF document map (Plotly) — with time bin selection
# ============================================================
# Hover over points to inspect titles/metadata.
#
# To select which time bins to display, set `SELECTED_BINS` to a list of
# time bin labels (e.g., ["-479–1679", "1679–1860"]).
# If `SELECTED_BINS` is None, all bins are shown.

import plotly.express as px

# ---- Prepare data ----
# Get all chronological bins from the original plot_df
all_bins = sorted(plot_df["time_bin"].dropna().unique(), key=get_start_year)

# Set this to a list of bins you want to keep, or None to keep all.
# To exclude the last (most recent) bin, uncomment the line below.
SELECTED_BINS = None   # e.g., all_bins[:3]

# Create a filtered copy if SELECTED_BINS is set
if SELECTED_BINS is not None:
    # Ensure the selected bins are in the data
    selected = [b for b in SELECTED_BINS if b in all_bins]
    if not selected:
        print("⚠️  No selected bins found in the data. Showing all bins.")
        plot_df_filtered = plot_df.copy()          # keep all
        time_bins = all_bins
    else:
        plot_df_filtered = plot_df[plot_df["time_bin"].isin(selected)].copy()
        time_bins = selected
else:
    plot_df_filtered = plot_df.copy()              # keep all
    time_bins = all_bins

# Prepare hover columns (from the original plot_df, which has all columns)
hover_cols = ["title"]
for c in ["author", "publication_year", "time_bin", "pg_id"]:
    if c in plot_df.columns:
        hover_cols.append(c)

# ---- Plot ----
fig = px.scatter(
    plot_df_filtered,                               # use the filtered copy
    x="svd1",
    y="svd2",
    color="time_bin",
    category_orders={"time_bin": time_bins},
    hover_data=hover_cols,
    title="Documents in TF–IDF space (interactive)",
    width=1000,
    height=700,
)

fig.update_traces(marker=dict(size=8, opacity=0.8))
fig.show()

In [ ]:
html_path = OUTPUT_DIR / "figures" / "nb04_tfidf_svd_interactive.html"

fig.write_html(html_path)

print("Saved interactive figure:", html_path)

In [ ]:
# ============================================================
# ADDITION 2 — Document similarity heatmap (TF–IDF cosine)
# ============================================================
# Goal:
# Visualize pairwise similarity between a small sample of documents.
#
# Interpretation:
# - bright blocks along the diagonal often indicate groups of similar texts
# - similarities across bins may suggest shared vocabulary/topics
#
# IMPORTANT:
# This is exploratory. Similarity reflects TF–IDF overlap, not “true meaning”.

# -----------------------------
# Sample documents (keep small for readability)
# -----------------------------
N_HEATMAP_DOCS = 40

rng = np.random.default_rng(42)
sample_idx = rng.choice(np.arange(len(df)), size=min(N_HEATMAP_DOCS, len(df)), replace=False)

# Sort sampled docs chronologically for easier reading
sample_df = df.iloc[sample_idx].copy()
sample_df = sample_df.sort_values(["time_bin", "publication_year"])
sample_df["publication_year"] = pd.to_numeric(sample_df["publication_year"], errors="coerce").astype("Int64")

plot_df["label"] = plot_df["pg_id"].astype(str) + " | " + plot_df["title"].fillna(plot_df["pg_id"])

sample_idx = sample_df.index.to_numpy()

# -----------------------------
# Compute cosine similarity
# -----------------------------
S_docs = cosine_similarity(T[sample_idx])

# Labels shown on the axes

labels = [
    f"{getattr(r, 'pg_id', r.title)} | {r.publication_year}"
    for r in sample_df.itertuples(index=False)
]

sim_df = pd.DataFrame(S_docs, index=labels, columns=labels)

# -----------------------------
# Plot heatmap
# -----------------------------
plt.figure(figsize=(12, 10))

sns.heatmap(
    sim_df,
    cmap="crest",
    xticklabels=True,
    yticklabels=True,
)

plt.title("Document similarity heatmap (TF–IDF cosine)")
plt.xlabel("Documents")
plt.ylabel("Documents")
plt.tight_layout()
plt.show()

# Inspect metadata of sampled docs
display(sample_df[["pg_id", "title", "publication_year"]].head(20))

### Interactive TF–IDF cosine similarity heatmap (Plotly)

In [ ]:
# Hover over cells to inspect:
# - title of document A
# - title of document B
# - time bins
# - cosine similarity

import plotly.graph_objects as go

# -----------------------------
# Sample documents
# -----------------------------
N_HEATMAP_DOCS = 25

## Random selection of texts in the corpus
# rng = np.random.default_rng(42)
# sample_idx = rng.choice(
#     np.arange(len(df)),
#     size=min(N_HEATMAP_DOCS, len(df)),
#     replace=False
# )

# sample_df = df.iloc[sample_idx].copy()

# Random selection of texts per time_bin
# Keep only rows with valid time bins
df_time = df.loc[df["time_bin"].notna()].copy()

# Stratified sample: same number of docs per bin
sample_parts = []

for _, g in df_time.groupby("time_bin", observed=True):
    sample_parts.append(
        g.sample(min(5, len(g)), random_state=42)
    )

sample_df = pd.concat(sample_parts, ignore_index=True)

# Sort chronologically for readability
sample_df = sample_df.sort_values(["time_bin", "publication_year"])

sample_idx = sample_df.index.to_numpy()

# -----------------------------
# Cosine similarity matrix
# -----------------------------
S_docs = cosine_similarity(T[sample_idx])

# -----------------------------
# Labels shown on axes
# -----------------------------
labels = [
    f"{getattr(r, 'title', r.pg_id)} | {r.publication_year}"
    for r in sample_df.itertuples(index=False)
]

# -----------------------------
# Hover text
# -----------------------------
hover_text = []

for i, r1 in enumerate(sample_df.itertuples(index=False)):
    row = []
    for j, r2 in enumerate(sample_df.itertuples(index=False)):
        row.append(
            f"""
            <b>Document A</b><br>
            {getattr(r1, 'title', r1.pg_id)}<br>
            Time bin: {r1.time_bin}<br>
            PG ID: {r1.pg_id}<br>
            <br>
            <b>Document B</b><br>
            {getattr(r2, 'title', r2.pg_id)}<br>
            Time bin: {r2.time_bin}<br>
            PG ID: {r2.pg_id}<br>
            <br>
            <b>Cosine similarity:</b> {S_docs[i, j]:.3f}
            """
        )
    hover_text.append(row)

# -----------------------------
# Plotly heatmap
# -----------------------------
fig = go.Figure(
    data=go.Heatmap(
        z=S_docs,
        x=labels,
        y=labels,
        colorscale="teal",
        zmin=float(np.min(S_docs)),
        zmax=1.0,
        text=hover_text,
        hoverinfo="text",
        colorbar=dict(title="Cosine similarity"),
    )
)

fig.update_layout(
    title="Interactive document similarity heatmap (TF–IDF cosine)",
    width=1000,
    height=900,
)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.update_xaxes(tickangle=90)

fig.show()

In [ ]:
# ------------------------------------------------------------
# Add nearest-neighbor information to sample_df
# ------------------------------------------------------------
# For each sampled document:
# - find the most similar OTHER document in the sample
# - store its title and cosine similarity

nearest_titles = []
nearest_sims = []

for i in range(len(sample_df)):
    sims = S_docs[i].copy()

    # Ignore self-similarity (always 1.0)
    sims[i] = -1

    j = np.argmax(sims)

    nearest_titles.append(
        getattr(sample_df.iloc[j], "title", sample_df.iloc[j]["pg_id"])
    )
    nearest_sims.append(float(sims[j]))

sample_df["nearest_neighbor"] = nearest_titles
sample_df["nearest_cosine"] = nearest_sims

# Optional: round for readability
sample_df["nearest_cosine"] = sample_df["nearest_cosine"].round(3)

display(
    sample_df[
        ["title", "time_bin", "nearest_neighbor", "nearest_cosine"]
    ].sort_values("nearest_cosine", ascending=False)#.head(20)
)

In [ ]:
display(
    sample_df[
        ["title", "time_bin", "nearest_neighbor", "nearest_cosine"]
    ].tail(20)
)

### Companion similarity table for the heatmap

In [ ]:
# Each row = one pair of sampled documents + their TF–IDF cosine similarity

pairs = []

for i in range(len(sample_df)):
    for j in range(i + 1, len(sample_df)):   # avoid duplicates + self-comparisons
        r1 = sample_df.iloc[i]
        r2 = sample_df.iloc[j]

        pairs.append({
            "cosine_similarity": float(S_docs[i, j]),
            "title_a": getattr(r1, "title", r1.pg_id),
            "time_bin_a": r1.time_bin,
            "pg_id_a": r1.pg_id,
            "title_b": getattr(r2, "title", r2.pg_id),
            "time_bin_b": r2.time_bin,
            "pg_id_b": r2.pg_id,
        })

sim_pairs = pd.DataFrame(pairs)

# Sort by strongest similarity first
sim_pairs = sim_pairs.sort_values("cosine_similarity", ascending=False)

print("Most similar document pairs (TF–IDF cosine):")
display(sim_pairs.head(30))

In [ ]:
html_path = OUTPUT_DIR / "figures" / "nb04-tfidf_similarity_heatmap_interactive.html"
fig.write_html(html_path)

print("\nSaved interactive heatmap:", html_path)

## Method notes

When you interpret patterns “over time,” always check:
- how many documents are in each bin
- whether year is a proxy (e.g., author death year)
- whether results are robust to a different binning choice

## Conclusion

In this notebook, we moved from basic corpus description toward more interpretive forms of textual analysis. Lexical diversity measures helped us reflect on vocabulary variation and on the limits of comparing texts of very different lengths. Association measures — collocations and keyness — showed how characteristic terms and phrase patterns can reveal historically or conceptually salient structures, while also reminding us that these outputs are sensitive to preprocessing, stopword choices, and corpus composition.

We then introduced **TF–IDF** as a first representational model of documents. Unlike raw frequency counts, TF–IDF highlights terms that are *distinctive* within documents or periods, allowing us to compare texts, identify nearest neighbours, and characterise time bins through their most informative vocabulary. This gives us an important bridge from descriptive corpus analysis to later modelling work: texts are no longer only counted, but represented as vectors that can be compared systematically.

TF–IDF remains a valuable baseline precisely because it is transparent, fast, and easy to interpret — but it carries a fundamental limitation. Because it represents each term as an independent dimension, it has no mechanism for capturing **semantic similarity**: words like *"computation"* and *"algorithm"* occupy entirely unrelated axes in a TF–IDF matrix, even though they are conceptually close. Embedding-based representations (which we will encounter in later notebooks) address this by learning dense vectors in which semantically related words are placed near one another, enabling richer notions of similarity, analogy, and meaning change. Where TF–IDF asks *"which words are distinctive?"*, embeddings ask *"which words behave alike?"* — a shift that opens qualitatively different analytical possibilities. We continue to use TF–IDF when interpretability and simplicity matter, but turn to embeddings when the research question requires sensitivity to meaning rather than mere occurrence.

Taken together, lexical diversity, association, and TF–IDF provide complementary lenses on the spread of ideas in the corpus. They do not yet tell us directly how concepts change in meaning, but they help us identify where patterns of vocabulary, emphasis, and similarity begin to differ across texts and periods. In the next notebooks, we will build on these foundations — first with structured linguistic annotation and conceptual-relation extraction, and then with richer representational models that can capture the semantic dimensions that TF–IDF, by design, leaves out.


In [ ]:
# -----------------------------
# Saved outputs (Session 5)
# -----------------------------
print("Wrote to:", OUTPUT_DIR)
print("Key files:")
for p in [
    "nb04-docs_with_time.parquet",
    "nb04-lexical_diversity_by_doc.csv",
    "nb04-lexical_diversity_by_time_bin.csv",
    "nb04-collocations_bigrams_top.csv",
    "nb04-keyness_top_terms_by_time_bin.csv",
    "nb04-tfidf_top_terms_by_time_bin.csv",
    "nb04-tfidf_terms.csv",
    "nb04-chunks.parquet"
]:
    fp = OUTPUT_DIR / p
    print(" -", p, "| exists:", fp.exists())

print("\nCache files:")
for p in [
    "nb04-tfidf_docs.npz",
]:
    fp = CACHE / p
    print(" -", p, "| exists:", fp.exists())

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A4a highlight;
```